- indexing 기능 : 방대한 데이터, 특정 데이터를 찾아오려고 할 때, 시간의 효율성 극대화 목적

- 인덱스 생성 
1) 단일 인덱스 : 인덱서의 조건이 1개
2) 복합 인덱스 : 인덱스 조건이 복합적인 요소로 구성
3) 다중키 인덱스 : 배열 안에 속해있는 값들을 인덱스의 조건으로 설정
4) 텍스트 인덱스 : 문자열을 활용한 인덱스 설정

- 인덱스 삭제 :
- 인덱스 리스트 조회 :


In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client.sample_mflix
movies = db.movies

In [2]:
# 단일 인덱스

result =  movies.create_index("title")
print(result)

# 인덱스의 작명 방법 (자동으로 되는거고, 사용자가 하는게 아님)
# 어떤 컬럼을 인덱스의 대상으로 했는가 (title)
# 정렬방향 (1 : 오름차순)

title_1


In [3]:
result =  movies.create_index([("title", -1)])
print(result)

# 디폴드 값이 오름차순이라 1이 되는거고
# 사용자가 내림차순으로 정의해주면 -1이 된다

title_-1


In [4]:
indexes =  movies.index_information()
print(indexes)

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'title_1': {'v': 2, 'key': [('title', 1)]}, 'title_-1': {'v': 2, 'key': [('title', -1)]}}


In [7]:
# 복합 인덱스

result = movies.create_index([("title", 1), ("year", -1)])
print(result)

title_1_year_-1


In [8]:
indexes =  movies.index_information()
print(indexes)

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'title_1': {'v': 2, 'key': [('title', 1)]}, 'title_-1': {'v': 2, 'key': [('title', -1)]}, 'title_1_year_-1': {'v': 2, 'key': [('title', 1), ('year', -1)]}}


In [10]:
for movie in movies.aggregate([
    {"$limit": 1}
]) :
    print(movie["cast"])

['Charles Kayser', 'John Ott']


In [11]:
# 배열 안에 입력된 값들 중 1개라도 있으면 보다 빠르게 값을 조회, 수집

result = movies.create_index("cast")
print(result)

cast_1


In [12]:
# 문자열로 구성된 컨텐츠 가운데, 해당 단어들이 속해져 있으면 값을 조회하고 수집

result = movies.create_index([{"plot", "text"}])
print(result)

plot_text


In [13]:
indexes =  movies.index_information()
print(indexes)

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'title_1': {'v': 2, 'key': [('title', 1)]}, 'title_-1': {'v': 2, 'key': [('title', -1)]}, 'title_1_year_-1': {'v': 2, 'key': [('title', 1), ('year', -1)]}, 'cast_1': {'v': 2, 'key': [('cast', 1)]}, 'plot_text': {'v': 2, 'key': [('_fts', 'text'), ('_ftsx', 1)], 'weights': SON([('plot', 1)]), 'default_language': 'english', 'language_override': 'language', 'textIndexVersion': 3}}


In [15]:
# 인덱스 삭제

movies.drop_indexes()

In [16]:
indexes =  movies.index_information()
print(indexes)

{'_id_': {'v': 2, 'key': [('_id', 1)]}}


In [ ]:
# 인덱스 세팅 시, 가중치 값을 부여 -> 특정 인덱스의 경우에 더 높은 점수를 부여한다
# 데이터 안에 값이 저장될때 (상품명, 상세페이지, 가격, 할인율 등 여러가지 값이 저장된다)
# 이 때 원하는 키워드를 조회해서 출력해달라고 한다면 같은 키워드가 여러 컬럼에 있다 하더라도 원하는 컬럼에 있는 값을 먼저 찾아오게 한다

In [18]:
movies.create_index([("title", "text"), ("plot", "text")], weights={"title": 5, "plot": 1})

# 값을 찾을때마다 가중치값을 연산하는게 아니라 상대적으로 중요하다, 아니다를 의미함
# 즉, 1점보다 5점이 더 중요하다는 의미 (5대 1 비율)

'title_text_plot_text'

In [19]:
indexes = movies.index_information()
print(indexes)

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'title_text_plot_text': {'v': 2, 'key': [('_fts', 'text'), ('_ftsx', 1)], 'weights': SON([('plot', 1), ('title', 5)]), 'default_language': 'english', 'language_override': 'language', 'textIndexVersion': 3}}


In [21]:
cursor = movies.find(
    {"$text": {"$search": "thriller"}},
    {"score": {"$meta": "textScore"}}
).sort([("score", {"$meta": "textScore"})]).limit(5)

print(cursor)

for doc in cursor :
    print(doc)

{'_id': ObjectId('573a1398f29313caabce9497'), 'plot': 'A night at the movies turns into a nightmare when Michael and his date are attacked by a hoard of bloody-thirsty zombies - only "Thriller" can save them now.', 'genres': ['Short', 'Horror', 'Music'], 'runtime': 13, 'rated': 'PG', 'cast': ['Michael Jackson', 'Ola Ray', 'Vincent Price', 'Hanala Sagal'], 'poster': 'https://m.media-amazon.com/images/M/MV5BODhhZjJlYTktZDQ2MS00Yzk4LWFlOTQtYTgyOGE1ZGE5YWEyL2ltYWdlXkEyXkFqcGdeQXVyMzA5MjgyMjI@._V1_SY1000_SX677_AL_.jpg', 'title': 'Michael Jackson: Thriller', 'fullplot': 'Michael Jackson and his date are watching a movie. They leave, and take a shortcut through the graveyard on the way home. Michael turns into a werepanther-type creature, and then later a zombie, as he gets down and funky in a tremendous dance scene to the tune of his song "Thriller."', 'languages': ['English'], 'released': datetime.datetime(1983, 12, 2, 0, 0), 'directors': ['John Landis'], 'writers': ['John Landis', 'Michael

In [ ]:
# 1) 해당 키워드가 많이 등장핳 수록 점수 획득을 많이 한다
# 2) 해당 키워드가 전체 데이터상 희소할 수록 검색 발견 시, 점수를 높게 측정 
#   (예를들어 100개의 영화 중에 스릴러가 90개면 점수가 낮게 매겨지고, 5개밖에 없으면 점수가 높게 매겨진다)
# 3) 필드 가중치 값을 통해서 값 부여 
# 이 1,2,3번의 값을 종합해서 최종적인 스코어 값을 매긴다